# 01 · Preparación de la base de modelado

**Proyecto:** Predicción del empleo adecuado en la provincia de Guayas mediante XGBoost y SHAP con factores de expansión de la ENEMDU 2025
**Autor:** Klever Merchán · Maestría en Inteligencia de Negocios y Ciencia de Datos, UDLA

Este notebook lleva la base anual de personas de la ENEMDU 2025 (INEC) hasta la base de modelado: filtra la PEA de Guayas, construye la variable objetivo, recodifica y construye las 15 predictoras, y guarda el resultado.

| | |
|---|---|
| **Entra** | `BDDenemdu_personas_2025_anual.sav` (INEC, ENEMDU Anual 2025) |
| **Sale** | `F1_flujo_muestra.csv`, `diccionario_2025.xlsx`, `base_modelado.pkl`, `base_modelado.csv` |

Se ejecuta de principio a fin con *Restart & Run All*. No contiene pasos aleatorios, así que no requiere semilla.

## 1. Configuración

Importaciones, ruta de la base y una función auxiliar para tasas ponderadas por `fexp`. La ruta se define una sola vez aquí.

In [1]:
import os
import numpy as np
import pandas as pd
import pyreadstat

RUTA_BASE = "Datos/BDDenemdu_personas_2025_anual.sav"


def resumen_ponderado(d, por):
    """Casos sin ponderar, % de la PEA ponderada y tasa ponderada de empleo adecuado por grupo."""
    g = d.assign(w=d['fexp'], wy=d['y'] * d['fexp']).groupby(por, observed=True)
    out = pd.DataFrame({
        'n': g.size(),
        'pct_pea': g['w'].sum() / d['fexp'].sum() * 100,
        'tasa_adecuado': g['wy'].sum() / g['w'].sum() * 100})
    return out.round(1)

## 2. Carga y verificación de la base

**Qué hace:** abre la base anual de personas y verifica que contenga las tres variables del diseño muestral (`upm`, `estrato`, `fexp`).
**Entra:** la `.sav` del INEC. **Sale:** `df25` (base nacional) y `meta25` (etiquetas).

In [2]:
df25, meta25 = pyreadstat.read_sav(RUTA_BASE)
print("Dimensiones de la base:", df25.shape)   # esperado: (334786, 139)

diseno = ['upm', 'estrato', 'fexp']
print("Variables del diseño muestral presentes:", {v: v in df25.columns for v in diseno})

Dimensiones de la base: (334786, 139)
Variables del diseño muestral presentes: {'upm': True, 'estrato': True, 'fexp': True}


In [3]:
print(df25['fexp'].describe().round(2))
print(f"\nPoblación nacional estimada (suma de fexp, todas las edades): {df25['fexp'].sum():,.0f}")

count    334786.00
mean         56.32
std         135.45
min           0.72
25%           9.88
50%          23.42
75%          55.54
max        4819.20
Name: fexp, dtype: float64

Población nacional estimada (suma de fexp, todas las edades): 18,855,042


## 3. Población de estudio y variable objetivo

**Qué hace:** obtiene la provincia desde el código de `ciudad`, filtra Guayas y conserva la PEA (`condact` 1–8 sin el 6, empleo no clasificado). Construye `y = 1` si la persona tiene empleo adecuado (`condact == 1`).
**Sale:** `gy` (todas las personas de Guayas), `pea25` (PEA de Guayas) y la tabla del flujo de la muestra (**Figura 1**).

In [4]:
df25['prov'] = df25['ciudad'].astype('Int64').astype(str).str.zfill(6).str[:2]
gy = df25[df25['prov'] == '09'].copy()

etq_condact = meta25.variable_value_labels.get('condact', {})
dist = gy['condact'].value_counts().sort_index()
dist.index = [f"{int(c)} · {etq_condact.get(c, '')}" for c in dist.index]
print("Condición de actividad en Guayas (casos sin ponderar):")
print(dist.to_string())

Condición de actividad en Guayas (casos sin ponderar):
0 · Menores de 15 años                                   9166
1 · Empleo Adecuado/Pleno                                9665
2 · Subempleo por insuficiencia de tiempo de trabajo     3920
3 · Subempleo por insuficiencia de ingresos                31
4 · Otro empleo no pleno                                 4825
5 · Empleo no remunerado                                  539
6 · Empleo no clasificado                                 189
7 · Desempleo abierto                                     507
8 · Desempleo oculto                                       38
9 · Población Económicamente Inactiva                   14939


In [5]:
pea25 = gy[gy['condact'].between(1, 8) & (gy['condact'] != 6)].copy()
pea25['y'] = (pea25['condact'] == 1).astype(int)

print("PEA de Guayas:", len(pea25))   # esperado: 19525
print("Suma de fexp en la PEA de Guayas:", round(pea25['fexp'].sum()))
print("\nBalance de y sin ponderar:")
print(pea25['y'].value_counts(normalize=True).round(3).to_string())
print("\nTasa ponderada de empleo adecuado:", round((pea25['y'] * pea25['fexp']).sum() / pea25['fexp'].sum(), 3))

PEA de Guayas: 19525
Suma de fexp en la PEA de Guayas: 1862300

Balance de y sin ponderar:
y
0    0.505
1    0.495

Tasa ponderada de empleo adecuado: 0.415


In [6]:
# Desempleados (condact 7 y 8) entre las personas fuera del empleo adecuado
fuera = pea25[pea25['y'] == 0]
desemp = fuera['condact'].isin([7, 8])
print("Sin ponderar:", desemp.sum(), "de", len(fuera), "=", round(desemp.mean() * 100, 1), "%")
print("Ponderado:", round(np.average(desemp, weights=fuera['fexp']) * 100, 1), "%")

Sin ponderar: 545 de 9860 = 5.5 %
Ponderado: 4.3 %


In [7]:
flujo = pd.DataFrame([
    ('Base nacional de personas', len(df25)),
    ('Personas de la provincia de Guayas', len(gy)),
    ('Excluidos: menores de 15 años (condact 0)', -(gy['condact'] == 0).sum()),
    ('Excluidos: población económicamente inactiva (condact 9)', -(gy['condact'] == 9).sum()),
    ('Excluidos: empleo no clasificado (condact 6)', -(gy['condact'] == 6).sum()),
    ('PEA de Guayas analizada', len(pea25))],
    columns=['paso', 'registros'])

# Control: Guayas menos las exclusiones debe dar exactamente la PEA analizada
assert len(gy) + flujo['registros'].iloc[2:5].sum() == len(pea25)

flujo.to_csv("F1_flujo_muestra.csv", index=False, encoding="utf-8-sig")
flujo

,paso,registros
0,Base nacional de personas,334786
1,Personas de la provincia de Guayas,43819
2,Excluidos: menores de 15 años (condact 0),-9166
3,Excluidos: población económicamente inactiva (...,-14939
4,Excluidos: empleo no clasificado (condact 6),-189
5,PEA de Guayas analizada,19525


## 4. Diccionario de variables

**Qué hace:** exporta el nombre, la etiqueta y las categorías de las 139 variables de la base. Es la fuente de la matriz de operacionalización.
**Sale:** `diccionario_2025.xlsx`.

In [8]:
dicc25 = pd.DataFrame({
    'variable': meta25.column_names,
    'etiqueta': [meta25.column_names_to_labels.get(c, "") for c in meta25.column_names],
    'categorias': [str(meta25.variable_value_labels.get(c, "")) for c in meta25.column_names]})

dicc25.to_excel("diccionario_2025.xlsx", index=False)
print("Diccionario:", dicc25.shape)   # esperado: (139, 3)

Diccionario: (139, 3)


## 5. Calidad de las variables candidatas

**Qué hace:** revisa faltantes, tipo y rango de las variables candidatas. Los faltantes de `p11` (sabe leer y escribir) y `p12a` (obtuvo título) no son errores sino saltos del cuestionario, como muestran los cruces con el nivel de instrucción.

**Decisiones que se justifican aquí:**
- `p11` se excluye: solo se pregunta a quienes tienen nivel Ninguno o Básica, y es redundante con instrucción y años de escolaridad.
- `p12a` se fusiona con `nnivins`: solo se pregunta en el nivel Superior y permite separar "Superior sin título" de "Superior con título".

In [9]:
candidatas = ['p02', 'p03', 'p04', 'p06', 'p15', 'p15aa', 'nnivins',
              'p07', 'p11', 'p12a', 'area', 'dominio']

calidad = pd.DataFrame({
    'etiqueta': [meta25.column_names_to_labels.get(v) for v in candidatas],
    'faltantes': pea25[candidatas].isna().sum(),
    'pct_faltantes': (pea25[candidatas].isna().mean() * 100).round(2),
    'categorias': pea25[candidatas].nunique(),
    'minimo': pea25[candidatas].min(),
    'maximo': pea25[candidatas].max()})
calidad

,etiqueta,faltantes,pct_faltantes,categorias,minimo,maximo
p02,Sexo,0,0.00,2,1.0,2.0
p03,Edad,0,0.00,77,15.0,92.0
p04,Relación de Parentesco,0,0.00,9,1.0,9.0
p06,Estado civil,0,0.00,6,1.0,6.0
p15,Como se considera,0,0.00,8,1.0,8.0
p15aa,Dónde nació,0,0.00,3,1.0,3.0
nnivins,Nivel de instrucción,0,0.00,4,1.0,5.0
p07,Asiste a clases,0,0.00,2,1.0,2.0
p11,Sabe leer y escribir,15196,77.83,2,1.0,2.0
p12a,Por sus estudios obtuvo algún título,13852,70.94,2,1.0,2.0


In [10]:
print("nnivins:", meta25.variable_value_labels.get('nnivins'))

print("\n--- p11 (sabe leer y escribir) según nivel de instrucción ---")
print(pd.crosstab(pea25['nnivins'], pea25['p11'].fillna('vacío'), margins=True))

print("\n--- p12a (obtuvo título) según nivel de instrucción ---")
print(pd.crosstab(pea25['nnivins'], pea25['p12a'].fillna('vacío'), margins=True))

nnivins: {1.0: 'Ninguno', 2.0: 'Centro de Alfabetización', 3.0: 'Educación Básica', 4.0: 'Educación Media/Bachillerato', 5.0: 'Superior'}

--- p11 (sabe leer y escribir) según nivel de instrucción ---
p11       1.0  2.0  vacío    All
nnivins                         
1.0         6  242      0    248
3.0      3886  195   1254   5335
4.0         0    0   8269   8269
5.0         0    0   5673   5673
All      3892  437  15196  19525

--- p12a (obtuvo título) según nivel de instrucción ---
p12a      1.0   2.0  vacío    All
nnivins                          
1.0         0     0    248    248
3.0         0     0   5335   5335
4.0         0     0   8269   8269
5.0      3929  1744      0   5673
All      3929  1744  13852  19525


## 6. Recodificaciones

**Qué hace:** agrupa categorías con pocos casos y construye la zona de residencia.

| Variable | Origen | Categorías |
|---|---|---|
| `parentesco` | `p04` | Jefe, Cónyuge, Hijo/a, Otros |
| `etnia` | `p15` | Mestizo, Montuvio, Afrodescendiente (afroecuatoriano, negro, mulato), Otros (indígena, blanco, otra) |
| `instruccion` | `nnivins` + `p12a` | Ninguno, Básica, Media/Bachillerato, Superior sin título, Superior con título |
| `zona_amp` | `dominio`, `area`, cantón | Guayaquil, Conurbación (Daule, Durán, Samborondón), Resto urbano, Resto rural |

Cualquier caso que no encaje queda como "SIN ASIGNAR" y se detecta en la verificación final.

In [11]:
pea25['parentesco'] = pea25['p04'].map({
    1: 'Jefe', 2: 'Cónyuge', 3: 'Hijo/a',
    4: 'Otros', 5: 'Otros', 6: 'Otros', 7: 'Otros', 8: 'Otros', 9: 'Otros'})

pea25['etnia'] = pea25['p15'].map({
    6: 'Mestizo', 5: 'Montuvio',
    2: 'Afrodescendiente', 3: 'Afrodescendiente', 4: 'Afrodescendiente',
    1: 'Otros', 7: 'Otros', 8: 'Otros'})

pea25['instruccion'] = np.select(
    [pea25['nnivins'] == 1,
     pea25['nnivins'] == 3,
     pea25['nnivins'] == 4,
     (pea25['nnivins'] == 5) & (pea25['p12a'] == 2),
     (pea25['nnivins'] == 5) & (pea25['p12a'] == 1)],
    ['Ninguno', 'Básica', 'Media/Bachillerato', 'Superior sin título', 'Superior con título'],
    default='SIN ASIGNAR')

for v in ['parentesco', 'etnia', 'instruccion']:
    print(f"\n=== {v} ===")
    print(pea25[v].value_counts(dropna=False).to_string())


=== parentesco ===
parentesco
Jefe       8752
Hijo/a     5441
Cónyuge    3021
Otros      2311

=== etnia ===
etnia
Mestizo             18208
Montuvio              565
Afrodescendiente      434
Otros                 318

=== instruccion ===
instruccion
Media/Bachillerato     8269
Básica                 5335
Superior con título    3929
Superior sin título    1744
Ninguno                 248


### 6.1 Zona de residencia

Primero se cruza dominio con área: el dominio Guayaquil es 100 % urbano, así que ambas variables son redundantes y se combinan en una zona de tres categorías. Luego la tabla por cantón muestra que Daule, Durán y Samborondón tienen tasas cercanas a Guayaquil y muy por encima del resto de la provincia; por eso forman la categoría Conurbación.

In [12]:
print(pd.crosstab(pea25['dominio'], pea25['area'], margins=True)
      .rename(index={2.0: 'Guayaquil', 6.0: 'Resto Guayas'},
              columns={1.0: 'Urbana', 2.0: 'Rural'}))

pea25['zona'] = np.select(
    [pea25['dominio'] == 2,
     (pea25['dominio'] == 6) & (pea25['area'] == 1),
     (pea25['dominio'] == 6) & (pea25['area'] == 2)],
    ['Guayaquil', 'Resto urbano', 'Resto rural'],
    default='SIN ASIGNAR')

area          Urbana  Rural    All
dominio                           
Guayaquil      16147      0  16147
Resto Guayas    1963   1415   3378
All            18110   1415  19525


In [13]:
nombres_canton = {
    '0901': 'Guayaquil', '0902': 'Alfredo Baquerizo Moreno', '0903': 'Balao',
    '0904': 'Balzar', '0905': 'Colimes', '0906': 'Daule', '0907': 'Durán',
    '0908': 'El Empalme', '0909': 'El Triunfo', '0910': 'Milagro',
    '0911': 'Naranjal', '0912': 'Naranjito', '0913': 'Palestina',
    '0914': 'Pedro Carbo', '0916': 'Samborondón', '0918': 'Santa Lucía',
    '0919': 'Salitre', '0920': 'San Jacinto de Yaguachi', '0921': 'Playas',
    '0922': 'Simón Bolívar', '0923': 'Coronel Marcelino Maridueña',
    '0924': 'Lomas de Sargentillo', '0925': 'Nobol',
    '0927': 'General Antonio Elizalde', '0928': 'Isidro Ayora'}

pea25['canton'] = pea25['ciudad'].astype('Int64').astype(str).str.zfill(6).str[:4]
pea25['canton_nombre'] = pea25['canton'].map(nombres_canton)
print("Cantones sin nombre:", pea25['canton_nombre'].isna().sum())   # esperado: 0

tabla_canton = (resumen_ponderado(pea25, 'canton')
                .join(pd.crosstab(pea25['canton'], pea25['zona'])))
tabla_canton.insert(0, 'nombre', tabla_canton.index.map(nombres_canton))
tabla_canton.sort_values('n', ascending=False)

Cantones sin nombre: 0


,nombre,n,pct_pea,tasa_adecuado,Guayaquil,Resto rural,Resto urbano
canton,,,,,,,
0901,Guayaquil,16294,63.1,49.1,16147,97,50
0907,Durán,564,6.8,45.3,0,53,511
0906,Daule,441,3.1,45.9,0,135,306
0908,El Empalme,355,4.0,19.4,0,199,156
0916,Samborondón,288,0.5,37.5,0,222,66
0919,Salitre,199,2.0,16.5,0,146,53
0910,Milagro,175,1.6,41.6,0,29,146
0911,Naranjal,170,1.6,38.4,0,68,102
0925,Nobol,117,1.1,21.6,0,0,117


In [14]:
conurbacion = ['0906', '0907', '0916']   # Daule, Durán, Samborondón

pea25['zona_amp'] = np.select(
    [pea25['zona'] == 'Guayaquil',
     pea25['canton'].isin(conurbacion),
     pea25['zona'] == 'Resto urbano',
     pea25['zona'] == 'Resto rural'],
    ['Guayaquil', 'Conurbación', 'Resto urbano', 'Resto rural'],
    default='SIN ASIGNAR')

resumen_ponderado(pea25, 'zona_amp')

,n,pct_pea,tasa_adecuado
zona_amp,,,
Conurbación,1293,10.4,45.1
Guayaquil,16147,61.7,49.4
Resto rural,1005,7.4,19.5
Resto urbano,1080,20.4,23.9


## 7. Variables construidas

**Qué hace:** construye los años de escolaridad, el nivel y el campo del título superior, y tres variables del hogar.

### 7.1 Años de escolaridad (`anios_esc`)

Se combinan el nivel (`p10a`) y el año aprobado (`p10b`). La tabla cruzada muestra qué años son posibles en cada nivel y justifica la fórmula:

| Nivel (`p10a`) | Años |
|---|---|
| Ninguno (1) | 0 |
| Primaria (4) | año aprobado |
| Educación Básica (5) | año aprobado − 1 |
| Secundaria (6) | 6 + año aprobado |
| Educación Media (7) | 9 + año aprobado |
| Superior no universitario y universitario (8, 9) | 12 + año aprobado |
| Postgrado (10) | 17 + año aprobado |

In [15]:
print("p10a:", meta25.variable_value_labels.get('p10a'))
print("\n--- Año aprobado (p10b) según nivel (p10a) ---")
print(pd.crosstab(pea25['p10a'], pea25['p10b'].fillna('vacío'), margins=True))

p10a: {1.0: ' Ninguno', 2.0: ' Centro de alfabetización', 3.0: ' Jardín de infantes', 4.0: ' Primaria', 5.0: ' Educación Básica', 6.0: ' Secundaria', 7.0: ' Educación  Media', 8.0: ' Superior no universitario', 9.0: ' Superior Universitario', 10.0: ' Post-grado'}

--- Año aprobado (p10b) según nivel (p10a) ---
p10b  1.0   2.0   3.0  4.0   5.0   6.0  7.0  8.0  9.0  10.0  vacío    All
p10a                                                                     
1.0     0     0     0    0     0     0    0    0    0     0    248    248
4.0    86   170   205  109   139  2799    0    0    0     0      0   3508
5.0     3     5     6    9    12     7  205   65   86   175      0    573
6.0   224   327   703  299   252  4508    0    0    0     0      0   6313
7.0   126   152  2932    0     0     0    0    0    0     0      0   3210
8.0    65   189   540    0     0     0    0    0    0     0      0    794
9.0   362   416   401  391  2655    76   15   41    0     0      0   4357
10.0   79   406    29 

In [16]:
p10a = pea25['p10a']
p10b = pea25['p10b']

pea25['anios_esc'] = np.select(
    [p10a == 1, p10a == 4, p10a == 5, p10a == 6, p10a == 7, p10a.isin([8, 9]), p10a == 10],
    [0, p10b, p10b - 1, 6 + p10b, 9 + p10b, 12 + p10b, 17 + p10b],
    default=np.nan)

print("Vacíos:", pea25['anios_esc'].isna().sum())   # esperado: 0
print(pea25.groupby('instruccion')['anios_esc'].agg(['min', 'mean', 'max']).round(1))

Vacíos: 0
                      min  mean   max
instruccion                          
Básica                0.0   6.3   9.0
Media/Bachillerato   10.0  11.8  12.0
Ninguno               0.0   0.0   0.0
Superior con título  13.0  16.9  21.0
Superior sin título  13.0  14.5  18.0


### 7.2 Nivel y campo del título superior (`tit_nivel`, `tit_campo`)

`p12b` es un código de 8 dígitos que solo existe para "Superior con título". El primer dígito es el nivel del título y los dos siguientes el campo amplio de la clasificación CINE. Los niveles 4, 5 y 6 se agrupan como Postgrado; los campos con pocos casos (05, 08 y 99) como "Otros campos".

In [17]:
print("--- ¿Quién tiene p12b? (True = vacío) ---")
print(pd.crosstab(pea25['instruccion'], pea25['p12b'].isna()))

cod = pea25['p12b'].dropna().astype(int).astype(str).str.zfill(8)
print("\nCasos por primer dígito (nivel):")
print(cod.str[0].value_counts().sort_index().to_string())
print("\nCasos por dígitos 2-3 (campo CINE):")
print(cod.str[1:3].value_counts().sort_index().to_string())

--- ¿Quién tiene p12b? (True = vacío) ---
p12b                 False  True 
instruccion                      
Básica                   0   5335
Media/Bachillerato       0   8269
Ninguno                  0    248
Superior con título   3929      0
Superior sin título      0   1744

Casos por primer dígito (nivel):
p12b
1     115
2     544
3    2769
4      61
5     435
6       5

Casos por dígitos 2-3 (campo CINE):
p12b
01     673
02     153
03     568
04    1028
05      49
06     256
07     520
08      66
09     394
10     213
99       9


In [18]:
nivel_map = {'1': 'Técnico', '2': 'Tecnológico', '3': 'Grado',
             '4': 'Postgrado', '5': 'Postgrado', '6': 'Postgrado'}

campo_map = {'01': 'Educación', '02': 'Artes y humanidades',
             '03': 'Ciencias sociales y derecho', '04': 'Administración',
             '06': 'TIC', '07': 'Ingeniería', '09': 'Salud', '10': 'Servicios',
             '05': 'Otros campos', '08': 'Otros campos', '99': 'Otros campos'}

pea25['tit_nivel'] = cod.str[0].map(nivel_map)
pea25['tit_campo'] = cod.str[1:3].map(campo_map)
pea25['tit_nivel'] = pea25['tit_nivel'].fillna('Sin título superior')
pea25['tit_campo'] = pea25['tit_campo'].fillna('Sin título superior')

for v in ['tit_nivel', 'tit_campo']:
    print(f"\n=== {v} ===")
    print(pea25[v].value_counts().to_string())


=== tit_nivel ===
tit_nivel
Sin título superior    15596
Grado                   2769
Tecnológico              544
Postgrado                501
Técnico                  115

=== tit_campo ===
tit_campo
Sin título superior            15596
Administración                  1028
Educación                        673
Ciencias sociales y derecho      568
Ingeniería                       520
Salud                            394
TIC                              256
Servicios                        213
Artes y humanidades              153
Otros campos                     124


### 7.3 Variables del hogar (`tam_hogar`, `menores5`, `mayores65`)

Se cuentan sobre **todas** las personas del hogar en Guayas (`gy`), no solo sobre la PEA.

Como la base anual acumula visitas del panel rotativo, se verifica que el identificador de hogar no mezcle visitas distintas. El conteo se hace por `id_hogar` **y** `periodo`, y se compara con el conteo solo por `id_hogar`. Si ambos coinciden, no hay doble conteo.

In [19]:
print("Personas en Guayas:", len(gy), "| Hogares (id_hogar):", gy['id_hogar'].nunique())
print("Combinaciones id_hogar + periodo:", len(gy[['id_hogar', 'periodo']].drop_duplicates()))

jefes = gy[gy['p04'] == 1].groupby('id_hogar').size()
print("\nJefes por hogar:")
print(jefes.reindex(gy['id_hogar'].unique(), fill_value=0).value_counts().sort_index().to_string())

Personas en Guayas: 43819 | Hogares (id_hogar): 12738
Combinaciones id_hogar + periodo: 12738

Jefes por hogar:
1    12738


In [20]:
miembros = gy.assign(menor5=gy['p03'] < 5, mayor65=gy['p03'] >= 65)
agregados = dict(tam_hogar=('id_persona', 'size'),
                 menores5=('menor5', 'sum'),
                 mayores65=('mayor65', 'sum'))

hogar_visita = miembros.groupby(['id_hogar', 'periodo']).agg(**agregados)
hogar_solo_id = miembros.groupby('id_hogar').agg(**agregados)

nuevas = ['tam_hogar', 'menores5', 'mayores65']
por_visita = pea25.join(hogar_visita, on=['id_hogar', 'periodo'])[nuevas]
por_id = pea25.join(hogar_solo_id, on='id_hogar')[nuevas]
print("¿El conteo por visita coincide con el conteo por id_hogar?", por_visita.equals(por_id))

pea25 = pea25.join(hogar_visita, on=['id_hogar', 'periodo'])
print("Vacíos:", pea25[nuevas].isna().sum().to_dict())
print(pea25[nuevas].describe().round(2))

¿El conteo por visita coincide con el conteo por id_hogar? True
Vacíos: {'tam_hogar': 0, 'menores5': 0, 'mayores65': 0}
       tam_hogar  menores5  mayores65
count   19525.00  19525.00   19525.00
mean        4.18      0.23       0.34
std         1.99      0.51       0.62
min         1.00      0.00       0.00
25%         3.00      0.00       0.00
50%         4.00      0.00       0.00
75%         5.00      0.00       1.00
max        16.00      6.00       5.00


## 8. Exportación de la base de modelado

**Qué hace:** selecciona las 15 predictoras y las columnas de apoyo, verifica que no haya vacíos ni categorías "SIN ASIGNAR", y compara con la versión anterior de la base si existe.
**Sale:** `base_modelado.pkl` (para los notebooks siguientes) y `base_modelado.csv` (legible en Excel o R).

Columnas de apoyo (no predictoras): `y` (objetivo), `fexp`, `upm` y `estrato` (diseño muestral), `canton_nombre`, `id_persona` y `periodo`.

In [21]:
predictoras = ['p02', 'p03', 'p06', 'p15aa', 'p07', 'parentesco', 'etnia',
               'instruccion', 'anios_esc', 'tit_nivel', 'tit_campo',
               'zona_amp', 'tam_hogar', 'menores5', 'mayores65']
apoyo = ['y', 'fexp', 'upm', 'estrato', 'canton_nombre', 'id_persona', 'periodo']

base = pea25[predictoras + apoyo].copy()

sin_asignar = (base.astype(str) == 'SIN ASIGNAR').sum().sum()
print("Dimensiones:", base.shape)                       # esperado: (19525, 22)
print("Vacíos totales:", base.isna().sum().sum())       # esperado: 0
print("Celdas 'SIN ASIGNAR':", sin_asignar)            # esperado: 0
assert base.shape == (19525, 22) and base.isna().sum().sum() == 0 and sin_asignar == 0

Dimensiones: (19525, 22)
Vacíos totales: 0
Celdas 'SIN ASIGNAR': 0


In [22]:
# Si ya existe una base anterior, se compara antes de reemplazarla
if os.path.exists("base_modelado.pkl"):
    anterior = pd.read_pickle("base_modelado.pkl")
    # Se comparan valores (como texto) para que una diferencia de tipo de dato no cuente como cambio
    distintas = [c for c in base.columns
                 if c not in anterior.columns
                 or not base[c].astype(str).equals(anterior[c].astype(str))]
    if base.index.equals(anterior.index) and not distintas:
        print("La base nueva es idéntica a la anterior.")
    else:
        print("ATENCIÓN: la base nueva difiere de la anterior en:", distintas)
        print("Se guarda como base_modelado_nueva.pkl y no se reemplaza la anterior.")
        base.to_pickle("base_modelado_nueva.pkl")
        raise SystemExit("Revisar diferencias antes de continuar.")

base.to_pickle("base_modelado.pkl")
base.to_csv("base_modelado.csv", index=False, encoding="utf-8-sig")
print("Guardados: base_modelado.pkl y base_modelado.csv")

Guardados: base_modelado.pkl y base_modelado.csv
